# 🟢 PaddleOCR LOCAL FAST V5 · RESUME

Corrige la detección del runtime local.

Ya **no depende de que el repo esté montado en `/content/work`**.
Detecta el entorno local por:
- Python 3.12;
- kernel WSL2;
- GPU visible.

Los archivos temporales, la caché de pip y los modelos viven bajo `/root`,
que en tu configuración Docker ya es persistente.

Versiones:
- PaddlePaddle GPU 3.2.0
- CUDA wheel cu126
- PaddleOCR 3.2.0

No usa venv y no reinicia el kernel.

Durante la instalación muestra:
- barra de descarga real;
- MB descargados / total;
- porcentaje;
- velocidad estimada por `tqdm`;
- consola de `pip`;
- heartbeat si `pip` queda silencioso.

### Cambio importante en V5

El wheel gigante de Paddle **ya no lo descarga pip**.

Se guarda persistentemente en:

`/root/.cache/paddle-wheels/`

La descarga usa `curl --continue-at -`, por lo que si falla a mitad de camino,
la próxima ejecución **reanuda** el mismo archivo en lugar de empezar desde cero.

Antes de instalar se valida que el `.whl` sea un ZIP/wheel íntegro.


In [ ]:
import sys, platform, subprocess, importlib.metadata as md, socket, pathlib

print("🟢 LOCAL FAST V3 · Paddle 3.2.0 / PaddleOCR 3.2.0")
print()

release = platform.release()
platform_text = platform.platform()
host = socket.gethostname()

print("Python:", sys.version)
print("Platform:", platform_text)
print("Kernel:", release)
print("Hostname:", host)
print("HOME:", pathlib.Path.home())
print()

# El runtime Docker local corre sobre WSL2.
is_wsl = ("microsoft" in release.lower()) or ("wsl" in release.lower())
is_py312 = sys.version_info[:2] == (3, 12)

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
gpu_text = gpu.stdout.strip()

print("GPU:", gpu_text or "No detectada")
print()

if not is_py312:
    raise RuntimeError(
        f"Este notebook espera Python 3.12 del runtime local. "
        f"Encontré {sys.version_info.major}.{sys.version_info.minor}."
    )

if not is_wsl:
    print("⚠️ El kernel no parece WSL2.")
    print("Esto podría ser Colab Cloud. Revisá que estés conectado al runtime local.")
    print("No voy a instalar nada automáticamente en esta celda.")
else:
    print("✅ Runtime WSL2 local detectado.")

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return "NO INSTALADO"

print()
print("=== PAQUETES ===")
for name in ["paddlepaddle-gpu","paddleocr","paddlex","torch","pillow","numpy"]:
    print(f"{name:20} {ver(name)}")

In [ ]:
import sys, os, pathlib, subprocess, importlib.metadata as md, platform
import zipfile, shutil, time

PADDLE = "3.2.0"
OCR = "3.2.0"

# URL oficial exacta para Python 3.12 / Linux x86_64 / CUDA 12.6.
PADDLE_WHEEL_URL = (
    "https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/"
    "paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl"
)

release = platform.release().lower()
if "microsoft" not in release and "wsl" not in release:
    raise RuntimeError(
        "ABORTADO: este kernel no parece WSL2/local. "
        "No voy a descargar Paddle por accidente en Colab Cloud."
    )

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"Este wheel es cp312 y requiere Python 3.12. "
        f"Encontré {sys.version_info.major}.{sys.version_info.minor}."
    )

HOME = pathlib.Path.home()
PIP_CACHE = HOME / ".cache" / "pip"
MODEL_CACHE = HOME / ".cache" / "paddlex"
WHEEL_CACHE = HOME / ".cache" / "paddle-wheels"

for d in (PIP_CACHE, MODEL_CACHE, WHEEL_CACHE):
    d.mkdir(parents=True, exist_ok=True)

os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE)
os.environ["PADDLE_PDX_CACHE_HOME"] = str(MODEL_CACHE)
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"

WHEEL = WHEEL_CACHE / "paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl"

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

def human_bytes(n):
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    while n >= 1024 and i < len(units) - 1:
        n /= 1024
        i += 1
    return f"{n:.2f} {units[i]}"

def run(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True)

def wheel_valid(path):
    if not path.exists() or path.stat().st_size == 0:
        return False, "archivo inexistente o vacío"
    try:
        with zipfile.ZipFile(path, "r") as z:
            bad = z.testzip()
            if bad is not None:
                return False, f"entrada ZIP corrupta: {bad}"
            names = z.namelist()
            if not any(name.endswith(".dist-info/METADATA") for name in names):
                return False, "no encontré METADATA de wheel"
        return True, "OK"
    except Exception as e:
        return False, repr(e)

print("=== CACHÉS PERSISTENTES ===")
print("pip:    ", PIP_CACHE)
print("modelos:", MODEL_CACHE)
print("wheel:  ", WHEEL_CACHE)
print()

if ver("paddlepaddle-gpu") == PADDLE:
    print("✅ PaddlePaddle GPU 3.2.0 ya está instalado.")
else:
    existing = ver("paddlepaddle-gpu")
    if existing:
        raise RuntimeError(
            f"Hay PaddlePaddle GPU {existing} instalado. "
            "No voy a mezclar versiones automáticamente."
        )

    # ------------------------------------------------------------
    # DESCARGA REANUDABLE DEL WHEEL GRANDE
    # ------------------------------------------------------------
    if WHEEL.exists():
        print(f"📦 Ya existe un archivo parcial/local: {WHEEL}")
        print(f"   Tamaño actual: {human_bytes(WHEEL.stat().st_size)}")
        print("   curl intentará CONTINUAR desde ese punto.")
    else:
        print("⬇️ Primera descarga del wheel de Paddle.")
        print("   Si se corta, la próxima ejecución continúa desde donde quedó.")

    print()
    print("=== DESCARGA REANUDABLE ===")
    print("Destino:", WHEEL)
    print()

    curl = shutil.which("curl")
    if not curl:
        raise RuntimeError("No encontré curl dentro del runtime Docker.")

    # --continue-at -  => reanuda desde el tamaño actual del archivo
    # --retry-all-errors => también reintenta errores transitorios
    # --progress-bar => barra real de curl
    run([
        curl,
        "--location",
        "--fail",
        "--retry", "20",
        "--retry-delay", "2",
        "--retry-all-errors",
        "--connect-timeout", "30",
        "--continue-at", "-",
        "--progress-bar",
        "--output", WHEEL,
        PADDLE_WHEEL_URL,
    ])

    print()
    print("Descarga terminada.")
    print("Tamaño:", human_bytes(WHEEL.stat().st_size))

    # ------------------------------------------------------------
    # VERIFICAR ANTES DE PIP INSTALL
    # ------------------------------------------------------------
    print()
    print("🔎 Verificando integridad del wheel...")
    ok, detail = wheel_valid(WHEEL)

    if not ok:
        print("❌ El wheel descargado todavía está corrupto/incompleto.")
        print("Detalle:", detail)
        print()
        print("NO lo borro automáticamente.")
        print("Volvé a ejecutar esta celda: curl intentará reanudarlo.")
        raise RuntimeError("Wheel incompleto/corrupto; conservar archivo para reanudar.")

    print("✅ Wheel válido.")

    # Instalar DESDE EL ARCHIVO LOCAL. pip ya no descarga el wheel gigante.
    print()
    print("=== INSTALANDO PADDLE DESDE CACHE LOCAL ===")
    run([
        sys.executable, "-m", "pip", "install",
        "--retries", "10",
        "--timeout", "180",
        str(WHEEL),
    ])

# PaddleOCR es mucho menor; dejamos que pip use su cache persistente.
if ver("paddleocr") == OCR:
    print("✅ PaddleOCR 3.2.0 ya está instalado.")
else:
    existing = ver("paddleocr")
    if existing:
        raise RuntimeError(
            f"Hay PaddleOCR {existing} instalado. "
            "No voy a mezclar versiones automáticamente."
        )

    print()
    print("=== INSTALANDO PADDLEOCR ===")
    run([
        sys.executable, "-m", "pip", "install",
        "--retries", "10",
        "--timeout", "180",
        f"paddleocr=={OCR}",
    ])

print()
print("✅ Instalación lista.")
print("NO reinicies el kernel.")
print("La siguiente celda hace el smoke test en un proceso Python nuevo.")

In [ ]:
import sys, subprocess, pathlib, os, platform

release = platform.release().lower()
if "microsoft" not in release and "wsl" not in release:
    raise RuntimeError("No parece el runtime local WSL2.")

dev_dir = pathlib.Path.home()/".cache"/"paddleocr-dev"
dev_dir.mkdir(parents=True, exist_ok=True)

worker_path = dev_dir/"paddle_smoke_worker_v3.py"
worker_path.write_text('\nimport os, sys, json\nfrom pathlib import Path\n\nos.environ.setdefault("PADDLE_PDX_MODEL_SOURCE", "BOS")\nos.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(Path.home()/".cache"/"paddlex"))\n\nprint("=== WORKER NUEVO ===", flush=True)\nprint("Python:", sys.version, flush=True)\n\nimport paddle\nprint("Paddle:", paddle.__version__, flush=True)\nprint("CUDA:", paddle.is_compiled_with_cuda(), flush=True)\nprint("GPU count:", paddle.device.cuda.device_count(), flush=True)\n\nif not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() < 1:\n    raise RuntimeError("Paddle no ve la GPU CUDA.")\n\npaddle.set_device("gpu:0")\ntry:\n    print("GPU:", paddle.device.cuda.get_device_name(), flush=True)\nexcept Exception:\n    print("GPU: gpu:0", flush=True)\n\nimport PIL\nfrom PIL import Image, ImageDraw\nprint("Pillow:", PIL.__version__, flush=True)\n\nfrom paddleocr import PaddleOCR\nimport paddleocr\nprint("PaddleOCR:", getattr(paddleocr, "__version__", "unknown"), flush=True)\n\ndev_dir = Path.home()/".cache"/"paddleocr-dev"\ndev_dir.mkdir(parents=True, exist_ok=True)\n\nimg_path = dev_dir/"paddle_smoke_es.png"\nimg = Image.new("RGB", (1400, 320), "white")\nImageDraw.Draw(img).text(\n    (50, 100),\n    "Histologia epitelio plano simple prueba OCR espanol 12345",\n    fill="black"\n)\nimg.save(img_path)\n\nprint("Imagen:", img_path, flush=True)\nprint("Inicializando OCR...", flush=True)\n\nocr = PaddleOCR(\n    lang="es",\n    device="gpu:0",\n    use_doc_orientation_classify=False,\n    use_doc_unwarping=False,\n    use_textline_orientation=False,\n)\n\nprint("Ejecutando OCR...", flush=True)\nresults = ocr.predict(str(img_path))\n\ntexts = []\nfor res in results:\n    d = getattr(res, "json", res)\n    if callable(d):\n        d = d()\n    if isinstance(d, dict) and "res" in d:\n        d = d["res"]\n    if isinstance(d, dict):\n        texts += [str(x) for x in d.get("rec_texts", [])]\n\nprint("Textos:", json.dumps(texts, ensure_ascii=False), flush=True)\n\nif not texts:\n    raise RuntimeError("No se reconoció texto.")\n\nprint("✅ SMOKE TEST COMPLETO", flush=True)\n', encoding="utf-8")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PADDLE_PDX_MODEL_SOURCE"] = "BOS"
env["PADDLE_PDX_CACHE_HOME"] = str(pathlib.Path.home()/".cache"/"paddlex")

print("Ejecutando worker fresco:")
print(worker_path)
print()

proc = subprocess.Popen(
    [sys.executable, str(worker_path)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    print(line, end="", flush=True)

rc = proc.wait()
if rc:
    raise RuntimeError(f"Worker falló con código {rc}")